In [2]:
## loading the necessary libraries
import pandas as pd
import numpy as np

In [5]:
#loading the dataset ratings
ratings = pd.read_csv('/Users/partha/AI_ML_Course/MLCourse/ml-100k/u.data',
                      sep='\t', names=['user_id', 'movie_id', 'rating'], 
                      usecols=range(3), encoding='ISO-8859-1')

#loading the dataset movies
movies = pd.read_csv('/Users/partha/AI_ML_Course/MLCourse/ml-100k/u.item', 
                     sep='|', names=['movie_id', 'title'], usecols=range(2), encoding='ISO-8859-1')

In [8]:
#lets merge the two datasets to get the movie titles along with the ratings
ratings = pd.merge(movies,ratings)
ratings.head()

,movie_id,title,user_id,rating
0,1,Toy Story (1995),308,4
1,1,Toy Story (1995),287,5
2,1,Toy Story (1995),148,4
3,1,Toy Story (1995),280,4
4,1,Toy Story (1995),66,3


In [9]:
#lets create a pivot table with movie titles as rows and user ids as columns, and the ratings as values
movieRatings = ratings.pivot_table(
    index='user_id',
    columns='title',
    values='rating'
)
movieRatings.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#before we create a correlation matrix, we need to find the number of ratings for each movie. 
#We will create a new dataframe called 'ratings' with the average rating and the number of ratings for each movie.
movieStats = ratings.groupby('title').agg({'rating': [np.size, np.mean]})
movieStats.head()

/var/folders/_k/tz1g6g6j10v_wxt_5fvf1zhh0000gn/T/ipykernel_14905/1152124585.py:3: FutureWarning: The provided callable <function mean at 0x10c350c10> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  movieStats = ratings.groupby('title').agg({'rating': [np.size, np.mean]})


rating          
                            size      mean
title                                     
'Til There Was You (1997)      9  2.333333
1-900 (1994)                   5  2.600000
101 Dalmatians (1996)        109  2.908257
12 Angry Men (1957)          125  4.344000
187 (1997)                    41  3.024390

In [12]:
#lets find the popular movies with more than 250 ratings
popularMovies = movieStats['rating']['size'] >= 250
PopularMovies = movieStats[popularMovies].sort_values([('rating', 'mean')], ascending=False)

In [ ]:
#using our pivot table, can find the correlation between the ratings of 1 movie and other movies. 
#example - 'Star Wars (1977)'

starWarsRatings = movieRatings['Star Wars (1977)']
#starWarsRatings.head()

#using the corrwith function, we can find the correlation between the Star Wars ratings and other movies
StarWarsSimilar = movieRatings.corrwith(starWarsRatings)
#lets drop the NaN values and create a new dataframe with the correlation values
StarWarsSimilar = StarWarsSimilar.dropna()
StarWarsSimilar = StarWarsSimilar.sort_values(ascending=False)
df = pd.DataFrame(StarWarsSimilar, columns=['Correlation'])
#df.head(5)


/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2845: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2704: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2704: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


In [18]:
# the output gives similar movies to StarWars, but now the correlation values are not very useful, 
# as they are based on very few ratings.
# we will now join the correlation values with the number of ratings for each movie, 
# and filter out the movies with less than 250 ratings.
# we already createda a popularMovies dataframe 
# with the number of ratings for each movie, we can use that to filter out the movies 
# with less than 250 ratings.

df_2 = df.join(movieStats['rating']['size'])
df_2 = df_2[df_2['size'] >= 250]
df_2 = df_2.sort_values('Correlation', ascending=False)

df_2.head(10)


,Correlation,size
title,,
Star Wars (1977),1.000000,584
"Empire Strikes Back, The (1980)",0.748353,368
Return of the Jedi (1983),0.672556,507
Raiders of the Lost Ark (1981),0.536117,420
Indiana Jones and the Last Crusade (1989),0.350107,331
L.A. Confidential (1997),0.319065,297
E.T. the Extra-Terrestrial (1982),0.303619,300
Back to the Future (1985),0.274839,350
Jaws (1975),0.265459,280


In [32]:
#lets create a function that takes a movie title as input and returns the 
# top 10 similar movies based on the correlation values and the number of ratings.
def find_similar_movies(movie_title):

    #using our pivot table, can find the correlation between the ratings of 1 movie and other movies. 
    #example - 'Star Wars (1977)'

    movie_titleRatings = movieRatings[movie_title]
    #starWarsRatings.head()

    #using the corrwith function, we can find the correlation between the Star Wars ratings and other movies
    movie_title_similar = movieRatings.corrwith(movie_titleRatings)
    #lets drop the NaN values and create a new dataframe with the correlation values
    movie_title_similar = movie_title_similar.dropna()
    movie_title_similar = movie_title_similar.sort_values(ascending=False)
    df_similar = pd.DataFrame(movie_title_similar, columns=['Correlation'])

    #merging the correlation values with the number of ratings for each movie,
    # and filter out the movies with less than 350 ratings.
    df_target = df_similar.join(movieStats['rating']['size'])
    df_target = df_target[df_target['size'] >= 100]
    df_target = df_target.sort_values('Correlation', ascending=False)

    return df_target.head(5)



In [33]:
find_similar_movies('Batman Returns (1992)')

/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2845: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2704: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2704: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/partha/AI_ML_Course/venv2/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,Correlation,size
title,,
Batman Returns (1992),1.000000,142
Michael (1996),0.577122,119
Fly Away Home (1996),0.550590,180
"Peacemaker, The (1997)",0.543066,136
Batman (1989),0.484477,201


In [34]:
def find_similar_movies_2(
    movie_title,
    n=5,
    min_common_users=20,
    min_movie_ratings=100,
    shrinkage=50
):
    if movie_title not in movieRatings.columns:
        raise KeyError(f"Movie not found: {movie_title}")

    target_ratings = movieRatings[movie_title]
    results = []

    for candidate_title in movieRatings.columns:
        if candidate_title == movie_title:
            continue

        candidate_ratings = movieRatings[candidate_title]
        common = pd.concat(
            [target_ratings, candidate_ratings],
            axis=1
        ).dropna()

        overlap = len(common)
        total_candidate_ratings = candidate_ratings.count()

        if overlap < min_common_users:
            continue
        if total_candidate_ratings < min_movie_ratings:
            continue

        correlation = common.iloc[:, 0].corr(common.iloc[:, 1])
        if pd.isna(correlation):
            continue

        # Reduce the influence of correlations based on few shared ratings.
        score = correlation * overlap / (overlap + shrinkage)

        results.append({
            "title": candidate_title,
            "correlation": correlation,
            "common_users": overlap,
            "score": score
        })

    return (
        pd.DataFrame(results)
        .sort_values("score", ascending=False)
        .head(n)
        .set_index("title")
    )

In [36]:
find_similar_movies_2("Toy Story (1995)")

,correlation,common_users,score
title,,,
"Craft, The (1996)",0.549100,82,0.341107
Beauty and the Beast (1991),0.442960,166,0.340423
"Lion King, The (1994)",0.426778,173,0.331088
Aladdin (1992),0.411731,183,0.323376
That Thing You Do! (1996),0.427936,142,0.316494
